# Disaster Evacuation Routing System — ML Analysis
**Dataset:** Forest Fire Dataset (UCI ML Repository — Cortez & Morais, 2007)
**File:** forest_fire_dataset.csv
**Models:** KNN · K-Means · Gaussian Naive Bayes · Neural Network

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import csv, math, random
from collections import Counter
from ml_models import (
    load_dataset, KNNClassifier, GaussianNaiveBayes,
    KMeansClustering, NeuralNetworkRegressor,
    train_test_split, accuracy_score, confusion_matrix, rmse, mae
)
from algorithms import astar
plt.style.use('dark_background')
C = {'safe':'#00ff88','mod':'#ff8c00','danger':'#ff3b3b','path':'#ffe566','nn':'#ff69b4','km':'#00cfff','nb':'#ffe566'}
data = load_dataset()
print(f'Dataset loaded: {len(data)} records')
print(f'Features: FFMC, DMC, DC, ISI, temp, RH, wind, rain')
print(f'Labels: danger_zone (0/1/2), route_risk (0/1), evac_time (min)')

## 1. Dataset Exploration

In [ ]:
from collections import Counter
dz = Counter(d['danger_zone'] for d in data)
rr = Counter(d['route_risk']  for d in data)
et = [d['evac_time'] for d in data]
print('Danger Zone distribution:', dict(dz))
print('Route Risk distribution: ', dict(rr))
print(f'Evac time — min:{min(et):.1f}  max:{max(et):.1f}  mean:{sum(et)/len(et):.1f}')
fig,axes=plt.subplots(1,3,figsize=(15,4))
fig.suptitle('Forest Fire Dataset — Label Distributions',fontsize=13,color='white')
ax=axes[0]
ax.bar(['SAFE','MODERATE','DANGER'],[dz[0],dz[1],dz[2]],color=[C['safe'],C['mod'],C['danger']],width=0.5)
ax.set_title('Danger Zone Classes',color='white'); ax.set_facecolor('#0a0f0a')
ax=axes[1]
ax.bar(['SAFE ROUTE','RISKY ROUTE'],[rr[0],rr[1]],color=[C['safe'],C['danger']],width=0.4)
ax.set_title('Route Risk Classes',color='white'); ax.set_facecolor('#0a0f0a')
ax=axes[2]
ax.hist(et,bins=20,color=C['nn'],edgecolor='#0a0f0a',alpha=0.85)
ax.set_title('Evacuation Time (min)',color='white'); ax.set_facecolor('#0a0f0a')
plt.tight_layout()
plt.savefig('fig_dataset.png',dpi=120,bbox_inches='tight',facecolor='#060b0f')
plt.show()
print('Saved fig_dataset.png')

## 2. Feature Correlation Heatmap

In [ ]:
import numpy as np
feature_names=['FFMC','DMC','DC','ISI','temp','RH','wind','rain']
X_all=np.array([d['features'] for d in data])
corr=np.corrcoef(X_all.T)
fig,ax=plt.subplots(figsize=(8,6))
sns.heatmap(corr,annot=True,fmt='.2f',cmap='coolwarm',
            xticklabels=feature_names,yticklabels=feature_names,ax=ax,center=0)
ax.set_title('Feature Correlation Matrix — Forest Fire Dataset',color='white')
plt.tight_layout()
plt.savefig('fig_correlation.png',dpi=120,bbox_inches='tight',facecolor='#060b0f')
plt.show()
print('Saved fig_correlation.png')

## 3. KNN — Zone Danger Classification

In [ ]:
X_knn=[d['features'] for d in data]; y_knn=[d['danger_zone'] for d in data]
Xtr,Xte,ytr,yte=train_test_split(X_knn,y_knn)
k_vals=[1,3,5,7,9,11]; k_accs=[]
for k in k_vals:
    m=KNNClassifier(k=k); m.fit(Xtr,ytr)
    k_accs.append(accuracy_score(yte,m.predict(Xte)))
    print(f'  k={k:2d}  accuracy={k_accs[-1]}%')
best_k=k_vals[k_accs.index(max(k_accs))]
print(f'Best k={best_k}  ({max(k_accs)}%)')
knn=KNNClassifier(k=best_k); knn.fit(Xtr,ytr)
knn_preds=knn.predict(Xte)
knn_acc=accuracy_score(yte,knn_preds)
knn_cm=confusion_matrix(yte,knn_preds,[0,1,2])
print(f'Final KNN accuracy: {knn_acc}%')

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(16,4))
fig.suptitle('KNN — Zone Danger Classification (Real Fire Data)',fontsize=13,color='white')
ax=axes[0]
ax.plot(k_vals,k_accs,'o-',color=C['safe'],lw=2,ms=7)
ax.axhline(80,color=C['danger'],ls='--',alpha=0.6,label='80% threshold')
ax.set_title('k vs Accuracy',color='white'); ax.set_facecolor('#0a0f0a'); ax.legend()
for x,y in zip(k_vals,k_accs): ax.text(x,y+1,f'{y}%',ha='center',fontsize=8,color='white')
ax=axes[1]
sns.heatmap(np.array(knn_cm),annot=True,fmt='d',cmap='Greens',
            xticklabels=['SAFE','MOD','DANGER'],yticklabels=['SAFE','MOD','DANGER'],ax=ax)
ax.set_title('Confusion Matrix',color='white')
ax=axes[2]
feat_names=['FFMC','DMC','DC','ISI','temp','RH','wind','rain']
imps=np.array(X_knn).std(axis=0); imps=imps/imps.sum()
colors=[C['danger'],C['mod'],C['safe'],C['km'],C['nn'],C['safe'],C['mod'],C['danger']]
axes[2].bar(feat_names,imps,color=colors)
axes[2].set_title('Feature Spread (normalised)',color='white'); axes[2].set_facecolor('#0a0f0a')
plt.tight_layout()
plt.savefig('fig_knn.png',dpi=120,bbox_inches='tight',facecolor='#060b0f')
plt.show()

## 4. K-Means Clustering

In [ ]:
random.seed(42)
population=[(random.randint(0,19),random.randint(0,19)) for _ in range(60)]
shelters=[(2,2),(10,18),(18,5)]
k_range=range(2,8); inertias=[]; sils=[]
for k in k_range:
    km=KMeansClustering(k=k); km.fit(population)
    inertias.append(km.inertia(population)); sils.append(km.silhouette_score_approx(population))
    print(f'  k={k}  inertia={inertias[-1]:.1f}  silhouette={sils[-1]}')
km3=KMeansClustering(k=3); km3.fit(population)
assign=km3.assign_to_shelters(shelters)
print(f'Silhouette (k=3): {km3.silhouette_score_approx(population)}')

In [ ]:
clr=['#00ff88','#00cfff','#ff69b4','#ff8c00','#ffe566','#bf5fff']
fig,axes=plt.subplots(1,3,figsize=(16,4))
fig.suptitle('K-Means — Population Evacuation Zones',fontsize=13,color='white')
axes[0].plot(list(k_range),inertias,'o-',color=C['km'],lw=2,ms=7)
axes[0].set_title('Elbow Method',color='white'); axes[0].set_facecolor('#0a0f0a')
axes[1].bar(list(k_range),sils,color=C['km'],alpha=0.8)
axes[1].set_title('Silhouette Scores',color='white'); axes[1].set_facecolor('#0a0f0a')
ax=axes[2]; pop=np.array(population)
for i in range(km3.k):
    mask=km3.labels_==i
    ax.scatter(pop[mask,1],pop[mask,0],c=clr[i],s=40,label=f'Zone {i+1}',zorder=3)
    c=km3.centroids[i]; ax.scatter(c[1],c[0],c=clr[i],s=180,marker='X',edgecolors='white',lw=1,zorder=4)
    if i in assign:
        s=assign[i]['shelter']; ax.plot([c[1],s[1]],[c[0],s[0]],'--',color=clr[i],alpha=0.5)
for s in shelters: ax.scatter(s[1],s[0],c='#bf5fff',s=200,marker='^',edgecolors='white',lw=1.5,zorder=5)
ax.set_title('Clusters & Shelter Assignment',color='white'); ax.set_facecolor('#0a0f0a'); ax.legend(fontsize=7)
plt.tight_layout()
plt.savefig('fig_kmeans.png',dpi=120,bbox_inches='tight',facecolor='#060b0f')
plt.show()

## 5. Naive Bayes — Route Safety

In [ ]:
X_nb=[d['features'] for d in data]; y_nb=[d['route_risk'] for d in data]
Xtr,Xte,ytr,yte=train_test_split(X_nb,y_nb)
nb=GaussianNaiveBayes(); nb.fit(Xtr,ytr)
nb_preds=nb.predict(Xte)
nb_acc=accuracy_score(yte,nb_preds); nb_cm=confusion_matrix(yte,nb_preds,[0,1])
print(f'Naive Bayes accuracy: {nb_acc}%')
for cls,name in [(0,'SAFE'),(1,'RISKY')]:
    tp=sum(1 for a,p in zip(yte,nb_preds) if a==cls and p==cls)
    fp=sum(1 for a,p in zip(yte,nb_preds) if a!=cls and p==cls)
    fn=sum(1 for a,p in zip(yte,nb_preds) if a==cls and p!=cls)
    pr=tp/(tp+fp) if (tp+fp) else 0; re=tp/(tp+fn) if (tp+fn) else 0
    f1=2*pr*re/(pr+re) if (pr+re) else 0
    print(f'  {name}: precision={pr:.3f} recall={re:.3f} f1={f1:.3f}')

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(16,4))
fig.suptitle('Naive Bayes — Route Safety (Real Fire Data)',fontsize=13,color='white')
sns.heatmap(np.array(nb_cm),annot=True,fmt='d',cmap='YlOrRd',
            xticklabels=['SAFE','RISKY'],yticklabels=['SAFE','RISKY'],ax=axes[0])
axes[0].set_title('Confusion Matrix',color='white')
safe_p=[nb.predict_proba(x)[0] for x,y in zip(Xte,yte) if y==0][:80]
risky_p=[nb.predict_proba(x)[0] for x,y in zip(Xte,yte) if y==1][:80]
axes[1].hist(safe_p,bins=15,alpha=0.75,color=C['safe'],label='Actual SAFE',density=True)
axes[1].hist(risky_p,bins=15,alpha=0.75,color=C['danger'],label='Actual RISKY',density=True)
axes[1].set_title('P(SAFE) Distribution',color='white'); axes[1].set_facecolor('#0a0f0a'); axes[1].legend()
feat=['FFMC','DMC','DC','ISI','temp','RH','wind','rain']
sm=nb.means_[0]/(nb.means_[0]+nb.means_[1]+1e-9)
rm_=nb.means_[1]/(nb.means_[0]+nb.means_[1]+1e-9)
x=np.arange(len(feat))
axes[2].bar(x-0.2,sm,0.35,label='SAFE',color=C['safe'],alpha=0.85)
axes[2].bar(x+0.2,rm_,0.35,label='RISKY',color=C['danger'],alpha=0.85)
axes[2].set_xticks(x); axes[2].set_xticklabels(feat,fontsize=8)
axes[2].set_title('Class Feature Means',color='white'); axes[2].set_facecolor('#0a0f0a'); axes[2].legend()
plt.tight_layout()
plt.savefig('fig_naive_bayes.png',dpi=120,bbox_inches='tight',facecolor='#060b0f')
plt.show()

## 6. Neural Network — Evacuation Time

In [ ]:
X_nn=[d['features'] for d in data]; y_nn=[d['evac_time'] for d in data]
Xtr,Xte,ytr,yte=train_test_split(X_nn,y_nn)
nn=NeuralNetworkRegressor(hidden1=16,hidden2=8,lr=0.01,epochs=200,batch_size=32)
nn.fit(Xtr,ytr)
nn_preds=[nn.predict([x]) for x in Xte]
nn_rmse=rmse(yte,nn_preds); nn_mae=mae(yte,nn_preds)
ss_r=sum((a-b)**2 for a,b in zip(yte,nn_preds))
ss_t=sum((a-sum(yte)/len(yte))**2 for a in yte)
r2=round(1-ss_r/ss_t,3) if ss_t>0 else 0
print(f'Neural Network: RMSE={nn_rmse} MAE={nn_mae} R²={r2}')

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(16,4))
fig.suptitle('Neural Network — Evacuation Time (Real Fire Data)',fontsize=13,color='white')
axes[0].plot(nn.loss_history,color=C['nn'],lw=1.5)
axes[0].set_title('Training Loss',color='white'); axes[0].set_facecolor('#0a0f0a')
axes[1].scatter(yte[:120],nn_preds[:120],alpha=0.5,color=C['nn'],s=20)
mn,mx=min(yte),max(yte); axes[1].plot([mn,mx],[mn,mx],'--',color='white',alpha=0.4)
axes[1].set_title(f'Actual vs Predicted (R²={r2})',color='white'); axes[1].set_facecolor('#0a0f0a')
res=[p-a for a,p in zip(yte,nn_preds)]
axes[2].hist(res,bins=20,color=C['nn'],edgecolor='#0a0f0a',alpha=0.85)
axes[2].axvline(0,color='white',ls='--',alpha=0.5)
axes[2].set_title('Residual Distribution',color='white'); axes[2].set_facecolor('#0a0f0a')
plt.tight_layout()
plt.savefig('fig_neural_net.png',dpi=120,bbox_inches='tight',facecolor='#060b0f')
plt.show()

## 7. A* Pathfinding Demo

In [ ]:
from matplotlib.colors import ListedColormap
random.seed(5)
g=[[0]*20 for _ in range(20)]
for r in range(20):
    for c in range(20):
        rnd=random.random()
        if rnd<0.15: g[r][c]=1
        elif rnd<0.25: g[r][c]=3
        elif rnd<0.30: g[r][c]=4
g[0][0]=g[19][19]=0
path,visited,cost=astar(g,(0,0),(19,19))
print(f'A*: path={len(path)} cost={cost:.2f} explored={len(visited)}')
dg=[row[:] for row in g]
for r,c in visited:
    if dg[r][c] in(0,4): dg[r][c]=7
for r,c in path:
    if dg[r][c] not in(1,3): dg[r][c]=6
dg[0][0]=2; dg[19][19]=5
cmap=ListedColormap(['#0a1208','#1e2030','#00ff88','#ff3b3b','#ff8c00','#00cfff','#ffe566','#0d2010'])
fig,ax=plt.subplots(figsize=(7,7))
ax.imshow(dg,cmap=cmap,vmin=0,vmax=7,interpolation='nearest')
patches=[mpatches.Patch(color=c,label=l) for c,l in [('#00ff88','Start'),('#00cfff','Goal'),('#ffe566','Path'),('#0d2010','Explored'),('#ff3b3b','Hazard'),('#ff8c00','Congested'),('#1e2030','Wall')]]
ax.legend(handles=patches,loc='upper right',fontsize=8)
ax.set_title(f'A* — Cost:{cost:.2f}  Path:{len(path)}  Explored:{len(visited)}',color='white')
ax.axis('off')
plt.tight_layout()
plt.savefig('fig_astar.png',dpi=120,bbox_inches='tight',facecolor='#060b0f')
plt.show()

## 8. Model Summary

In [ ]:
print('='*55)
print('  MODEL PERFORMANCE SUMMARY — REAL FIRE DATASET')
print('='*55)
print(f'  KNN (k={best_k})        Accuracy: {knn_acc}%')
print(f'  Naive Bayes         Accuracy: {nb_acc}%')
print(f'  Neural Network      R²: {r2}  RMSE: {nn_rmse} min')
print('='*55)
print(f'  KNN  ≥80%: {"PASS ✓" if knn_acc>=80 else "FAIL"}')
print(f'  NB   ≥80%: {"PASS ✓" if nb_acc>=80 else "FAIL"}')
fig,ax=plt.subplots(figsize=(7,4))
models=[f'KNN\n(k={best_k})','Naive\nBayes']
accs=[knn_acc,nb_acc]
colors=[C['safe'] if a>=80 else C['danger'] for a in accs]
bars=ax.bar(models,accs,color=colors,width=0.4)
ax.axhline(80,color=C['danger'],ls='--',lw=1.5,label='80% requirement')
ax.set_ylim(0,110); ax.set_title('Classification Accuracy — Real Fire Dataset',color='white')
ax.set_facecolor('#0a0f0a'); ax.legend()
for bar,v in zip(bars,accs): ax.text(bar.get_x()+bar.get_width()/2,v+1,f'{v}%',ha='center',fontweight='bold',color='white')
plt.tight_layout()
plt.savefig('fig_summary.png',dpi=120,bbox_inches='tight',facecolor='#060b0f')
plt.show()